# Centauri Bloom — post-training step 1: supervised fine-tune (SFT)

Starts from the **newest pretraining checkpoint on `Centauri-Bloom/Bloom-Text-0.2`** (the open PR with the most cumulative tokens, pinned to its exact commit), fine-tunes on the **Instruct mix v2** (cleaned Jaron v2 + Dolly + OASST1 + GSM8K train), scores parent and candidate on the locked eval, and uploads the result **only as a new pull request** on `Centauri-Bloom/Bloom-Chat-0.2`. `model.safetensors` on `main` is never replaced; merging is a human step.

**Before you run (Kaggle):**
1. Settings → Accelerator **GPU T4**, Internet **on**, and **Visibility: Private**. The run folder holds tokenized training windows until the last cell scrubs them.
2. Add the **private** Jaron dataset as an input. It must contain `bloom_instruction_expanded_v2.json` (sha256 `395fbe80…88b0`); the older 28,121-row file is also accepted and de-duplicated.
3. Enable the `HF_TOKEN` secret (a write token for the `Centauri-Bloom` org).
4. Run all. About 25–45 minutes on a T4 for 3,000 steps.

**Privacy:** Jaron's data is read only from the attached private input. It is never uploaded to GitHub or Hugging Face: the PR carries only weights, config, tokenizer, metrics, hashes/counts (`data-provenance.json`), evaluation, and a GGUF export of those weights.

In [ ]:
#@title 1. Settings
STEPS = 3000 #@param [1000,2000,3000,4000,5000] {type:"raw"}
#@markdown 3,000 steps is about two passes over the 48,000-example mix (32 examples per step).
PARENT_HF_REVISION = "auto" #@param {type:"string"}
#@markdown `auto` = newest Bloom-Text-0.2 PR by cumulative tokens (pinned to its commit). Or set e.g. `refs/pr/7`.
MIN_PARENT_TOKENS = 1_100_000_000 #@param {type:"raw"}
#@markdown Refuse to start unless the parent has at least this many cumulative tokens (pretraining session 6 should land at about 1.15B). Set 0 to run on an older PR on purpose.
PARENT_RUN_DIR = "" #@param {type:"string"}
#@markdown Optional: a finished pretraining run folder instead of Hugging Face. Leave blank normally.
JARON_FILE = "" #@param {type:"string"}
#@markdown Optional exact path(s) to the Jaron JSON; blank auto-detects reviewed files under /kaggle/input.
RUN_LOCKED_EVAL = True #@param {type:"boolean"}
UPLOAD = True #@param {type:"boolean"}
#@markdown Upload opens a PR only if the candidate's validation loss is not worse than the parent's (override below).
FORCE_UPLOAD = False #@param {type:"boolean"}
REPO_REF = "post-training-sft-rl" #@param {type:"string"}
#@markdown Git ref of centauri-bloom to run. Use `main` once the post-training PR is merged.
CONFIRM_PRIVATE_NOTEBOOK = False #@param {type:"boolean"}
#@markdown Tick only after confirming this notebook is Private and the Jaron dataset is a private input.

DATASET = "Instruct mix v2 (Jaron v2 + open)"
TEXT_REPO = "Centauri-Bloom/Bloom-Text-0.2"
CHAT_REPO = "Centauri-Bloom/Bloom-Chat-0.2"
if not CONFIRM_PRIVATE_NOTEBOOK:
    raise ValueError("Confirm the notebook is Private and the Jaron dataset is a private input before running.")
if type(STEPS) is not int or not 100 <= STEPS <= 10000:
    raise ValueError("STEPS must be an integer from 100 to 10,000.")
print("SFT:", DATASET, "|", STEPS, "steps | parent:", PARENT_RUN_DIR or PARENT_HF_REVISION, "| target PR on", CHAT_REPO)

In [ ]:
#@title 2. Set up the environment
import os, sys, subprocess, json, shutil
from pathlib import Path
from datetime import datetime, timezone

IN_KAGGLE = Path("/kaggle/working").is_dir()
os.environ["HF_HUB_DISABLE_XET"] = "1"

def checked(args):
    p = subprocess.run(args, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    print(p.stdout, flush=True)
    if p.returncode:
        raise RuntimeError(f"Setup failed (exit {p.returncode}). See the full diagnostic above.")

def find_repo(start):
    cur = Path(start).resolve()
    for p in [cur, *cur.parents]:
        if (p / "pyproject.toml").is_file() and (p / "bloom" / "model.py").is_file():
            return p
    return None

if IN_KAGGLE:
    REPO = Path("/kaggle/working/centauri-bloom")
    REF = (REPO_REF or "main").strip() or "main"
    if not (REPO / ".git").exists():
        checked(["git", "clone", "--depth", "1", "--branch", REF, "https://github.com/InterstellarNerd8760/centauri-bloom.git", str(REPO)])
    else:
        checked(["git", "-C", str(REPO), "fetch", "--depth", "1", "origin", REF])
        checked(["git", "-C", str(REPO), "checkout", "-B", REF, "FETCH_HEAD"])
    BASE = Path("/kaggle/working/centauri-bloom-runs")
else:
    REPO = find_repo(Path.cwd())
    if REPO is None:
        raise RuntimeError("Run this on Kaggle, or from a centauri-bloom checkout.")
    BASE = REPO / "runs"
os.chdir(REPO)
BASE.mkdir(parents=True, exist_ok=True)
# Keep the preinstalled CUDA PyTorch; add only what the trainer needs.
checked([sys.executable, "-m", "pip", "install", "--no-deps", "-e", "."])
checked([sys.executable, "-m", "pip", "install", "tokenizers", "datasets", "huggingface_hub", "safetensors", "gguf"])

from bloom.notebook_workflow import submit_candidate, sha256, pick_device, preflight, upload_base_revision
DEVICE = pick_device()
if IN_KAGGLE and DEVICE != "cuda":
    raise RuntimeError("Enable a GPU accelerator (T4) in the Kaggle notebook settings, then restart.")
if DEVICE == "cuda" and shutil.which("nvidia-smi"):
    checked(["nvidia-smi"])
preflight(DEVICE)

In [ ]:
#@title 3. Authorize Hugging Face (candidate PR only)
from getpass import getpass
from bloom.notebook_workflow import auth_client
try:
    if IN_KAGGLE:
        from kaggle_secrets import UserSecretsClient
        token = UserSecretsClient().get_secret("HF_TOKEN")
    else:
        token = os.environ.get("HF_TOKEN")
except Exception:
    token = None
if not token:
    token = getpass("Hugging Face write token: ")
AUTH_API = auth_client(CHAT_REPO, token, require_repo=True)
del token
print("Authenticated. This notebook can only open a PR on", CHAT_REPO, "- it never writes to main and never creates repos.")

from bloom.posttrain import newest_pretrain
if PARENT_RUN_DIR:
    PARENT_REV, PARENT_INFO = None, {"run_dir": PARENT_RUN_DIR}
elif PARENT_HF_REVISION.strip() in ("", "auto"):
    PARENT_REV, PARENT_INFO = newest_pretrain(AUTH_API, TEXT_REPO, token=AUTH_API.token, min_tokens=MIN_PARENT_TOKENS)
else:
    PARENT_REV, PARENT_INFO = PARENT_HF_REVISION.strip(), {"revision": PARENT_HF_REVISION.strip(), "note": "explicit"}
print("Parent:", json.dumps(PARENT_INFO))

In [ ]:
#@title 4. Fine-tune (reads Jaron from the private Kaggle input only)
from bloom.trainer_data import KNOWN_JARON_SHAS
def reviewed(path):
    return Path(path).is_file() and sha256(path) in KNOWN_JARON_SHAS
if JARON_FILE:
    jaron_arg = [p.strip() for p in JARON_FILE.split(",") if p.strip()]
elif IN_KAGGLE:
    jaron_arg = [str(p) for p in Path("/kaggle/input").rglob("*.json") if reviewed(p)]
else:
    default = Path.home() / "Downloads" / "bloom_instruction_expanded_v2.json"
    jaron_arg = [str(default)] if reviewed(default) else []
if not jaron_arg:
    raise ValueError("No reviewed Jaron JSON found. Attach the private dataset (it must include the v2 file) or set JARON_FILE.")
bad = [p for p in jaron_arg if not reviewed(p)]
if bad:
    raise ValueError("Not an allowlisted Jaron file: " + ", ".join(Path(p).name for p in bad))
print("Jaron inputs (names only):", [Path(p).name for p in jaron_arg])

OUT_DIR = BASE / ("bloom-50m-post-sft-" + datetime.now(timezone.utc).strftime("%Y%m%d-%H%M%S"))
payload = json.dumps(dict(recipe=DATASET, steps=STEPS, repo_dir=str(REPO), out_dir=str(OUT_DIR), jaron_path=jaron_arg,
                          recover=False, model_size="50M", device=DEVICE, parent_run_dir=PARENT_RUN_DIR or None,
                          parent_hf_repo=None if PARENT_RUN_DIR else TEXT_REPO, parent_hf_revision=PARENT_REV,
                          stage="sft", locked_eval=bool(RUN_LOCKED_EVAL)))
script = ("import json,sys,os; from huggingface_hub import HfApi; from bloom.notebook_workflow import preflight; "
          "from bloom.trainer import run_training; args=json.loads(sys.argv[1]); preflight(args['device']); "
          "run_training(HfApi(token=os.environ['HF_TOKEN']), **args)")
env = os.environ.copy()
env["HF_TOKEN"] = AUTH_API.token
try:
    result = subprocess.run([sys.executable, "-u", "-c", script, payload], env=env)
finally:
    env.pop("HF_TOKEN", None)
if result.returncode:
    raise RuntimeError("Training failed; see the log above. Partial checkpoints are in " + str(OUT_DIR))
metrics = json.loads((OUT_DIR / "metrics.json").read_text())
print("Run folder:", OUT_DIR)

In [ ]:
#@title 5. Compare candidate with parent
ev = json.loads((OUT_DIR / "evaluation.json").read_text())
print("Validation loss (assistant tokens, same windows): parent %.4f -> candidate %.4f" % (
    metrics["parent_val_loss_same_windows"], metrics["val_loss"]))
print("Per-source validation loss:", json.dumps(metrics.get("source_val_losses"), indent=1))
for key in ("parent_locked_eval", "candidate_locked_eval"):
    if metrics.get(key):
        m = metrics[key]
        print(key, "| arithmetic exact:", m["arithmetic_exact"], "| format:", m["format_rate"], "| repetition:", m["repetition_rate"],
              "| 12x5:", (m.get("times_12_5") or {}).get("text"))
print(json.dumps(ev["candidate_samples"], indent=1))

In [ ]:
#@title 6. Open the candidate PR (never main), then scrub tokenized private data
from bloom.posttrain import assert_upload_clean, scrub_private_artifacts, provenance_is_private_safe
if AUTH_API is None:
    raise RuntimeError("Authorize first.")
prov = json.loads((OUT_DIR / "data-provenance.json").read_text())
if not provenance_is_private_safe(prov):
    raise RuntimeError("data-provenance.json contains raw example fields; refusing to upload.")
worse = metrics["val_loss"] > metrics["parent_val_loss_same_windows"]
if UPLOAD and worse and not FORCE_UPLOAD:
    print("Candidate validation loss is worse than the parent's; not uploading. Set FORCE_UPLOAD to override.")
elif UPLOAD:
    assert_upload_clean(["model.safetensors", "config.json", "tokenizer.json", "metrics.json",
                         "data-provenance.json", "evaluation.json", "bloom-f32.gguf"])
    if metrics["additional_steps"] != STEPS or metrics["dataset"] != DATASET or metrics["target_repo"] != CHAT_REPO:
        raise RuntimeError("Run settings differ from the completed candidate.")
    submit_candidate(AUTH_API, CHAT_REPO, OUT_DIR, STEPS, upload_base_revision(metrics), metrics["run_name"])
    print("A PR was opened on", CHAT_REPO, "- review it and merge it yourself. Main was not touched.")
else:
    print("UPLOAD is off; nothing was sent to Hugging Face.")
print("Scrubbed:", scrub_private_artifacts(OUT_DIR))